In [ ]:
# Importing Libraries
import os
import requests

from dotenv import load_dotenv
from bs4 import BeautifulSoup

from IPython.display import Markdown, display
from openai import OpenAI


In [ ]:
# Load environment variables in a file called .env

load_dotenv(override=True)

# MODEL CONFIGURATION
MODEL_NAME = os.getenv("MODEL_NAME", "llama3.2")
MODEL_BASE_URL = os.getenv("MODEL_BASE_URL", "http://localhost:11434/v1")

# OpenAI api key
api_key = os.getenv("OPENAI_API_KEY")
if not api_key:
    print("API Key not found")
elif not api_key.startswith("sk-proj-"):
    print("an api key was found but it doesn't start with sk-proj-")
elif api_key.strip() != api_key:
    print("an api key was found but it looks like it might have space or tab characters at the start or end - please remove them")
else:
    print("API Key found and looks good so far")


In [ ]:
# Read openai toggle
SHOULD_USE_OPENAI_FLAG = os.getenv("SHOULD_USE_OPENAI_FLAG")

if SHOULD_USE_OPENAI_FLAG == "False":
    api_key = "model_api_key"

In [ ]:
# Create openapi object
openai = OpenAI(base_url=MODEL_BASE_URL, api_key=api_key)

In [ ]:
def call_openai_and_read_response(incoming_messages):
    openai_response = openai.chat.completions.create(
        model=MODEL_NAME,
        messages=incoming_messages
    )

    return openai_response.choices[0].message.content

# Testing openapi call - TEST1
message = "Hello! This is my first ever message to you! Hi, tell me your model name and actual name!"
messages = [{"role": "user", "content": message}]
Markdown(call_openai_and_read_response(messages))

In [ ]:
# Testing openapi call - TEST2
messages = [
    {"role": "system", "content": "You are a helpful assistant"},
    {"role": "user", "content": "What is 2334 + 2?"}
]
Markdown(call_openai_and_read_response(messages))

In [ ]:
# Testing openapi call - TEST3
messages = [
    {"role": "system", "content": "You are a NOT a helpful assistant"},
    {"role": "user", "content": "What is 2334 + 2?"}
]
Markdown(call_openai_and_read_response(messages))

## Assignment - Mini project

In [ ]:
# A class to represent a Webpage
# If you're not familiar with Classes, check out the "Intermediate Python" notebook

# Some websites need you to use proper headers when fetching them:
headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/117.0.0.0 Safari/537.36"
}

class Website:
    def __init__(self, url):
        """
        Create this Website object from the given url using the BeautifulSoup library
        """
        self.url = url
        response = requests.get(url, headers=headers)
        soup = BeautifulSoup(response.content, "html.parser")
        self.title = soup.title.string if soup.title else "No title found"
        for irrelevant in soup.body(["script", "style", "img", "input"]):
            irrelevant.decompose()
        self.text = soup.body.get_text(separator="\n", strip=True)

In [ ]:
# Step 1: Creating system and user prompts
system_prompt = "You're a smart humouros news assitant. \
You prefer humanity above everything else. \
You are not biased towards any race \
You believe in simplyfying complex things \
You ignore racial abuses, provides short summary in points, ignoring texts that might be a navigation related \
You respond in markdown"

user_prompt = """I want to know the history of the ai evolution from start to latest year 2026"""

# Step 2: Make the messages list
messages = [
    {
        "role": "system", "content": system_prompt
    },
    {
        "role": "user", "content": user_prompt
    }
]
# Step 3: Call OpenAI - using native function
response = openai.chat.completions.create(model=MODEL_NAME, messages=messages)

# Step 4: print the result
Markdown(response.choices[0].message.content)

## Another toy porject - Privacy Policy Summarizer

In [ ]:
# A function that writes a User Prompt that asks for summaries of websites
def user_prompt_for(website):
    user_prompt = f"You are looking at a website titled {website.title}"
    user_prompt += "\nThe contents of this website is as follows; \
please provide a short summary of this website in markdown. \
If it includes news or announcements, then summarize these too.\n\n"
    user_prompt += website.text
    return user_prompt


In [ ]:
system_prompt = """You are an expert consumer policy analyst.
Analyze the provided Terms & Conditions, Terms of Service, Privacy Policy, or similar legal policy and explain what an ordinary user should know before accepting it.
Ignore navigation, advertisements, menus, footers, and unrelated webpage content.

Focus on:
- What data is collected and how it is used
- Data sharing, tracking, and retention
- Fees, subscriptions, renewals, cancellations, and refunds
- User responsibilities and restrictions
- Company's rights to suspend or terminate accounts
- User content and intellectual property
- Liability and disclaimers
- Arbitration, disputes, and governing law
- Any unusual or potentially unfavorable clauses
- Anything important that a user might easily overlook

Do not invent information. If something is unclear or not specified, say so.
Do not claim that a clause is illegal or unlawful. Provide informational analysis, not legal advice.
Prioritize important clauses over generic legal boilerplate.
Explain legal language in simple terms and clearly distinguish what the policy explicitly states from your interpretation of why it matters.
Respond in markdown. Do not wrap the markdown in a code block.

Use this structure:
# Policy Analysis
## Summary
Give a concise overview of what the policy means for the user.
## Key Points
List the 5 to 8 most important things the user should know.
## What to Watch

Highlight the 3 to 5 clauses that deserve the most attention.
For each:
Clause:
Meaning:
Why it matters:

## Privacy
Summarize data collection, usage, sharing, tracking, retention, and user controls.

## User Risks
Assess the following as Low / Medium / High / Unclear:
- Privacy
- Financial
- Account restrictions
- Legal/dispute
- Overall concern

Briefly explain each assessment.

## Bottom Line
In 2 to 4 sentences, explain what the user should understand before accepting the policy."""

In [ ]:
def messages_for(website):
    return [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt_for(website)}
    ]


In [ ]:
def summarize(url):
    website = Website(url)
    response = openai.chat.completions.create(
        model = MODEL_NAME,
        messages = messages_for(website)
    )
    return response.choices[0].message.content

In [ ]:
# A function to display this nicely in the Jupyter output, using markdown

def display_summary(url):
    summary = summarize(url)
    display(Markdown(summary))

In [ ]:
display_summary("https://openai.com/policies/privacy-policy/")

In [ ]:
display_summary("https://www.spotify.com/in-en/legal/end-user-agreement/")